In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import scipy.integrate  as integrate 
import extcurves
from scipy.optimize import bisect

from astropy import constants
eV  = 1.60218e-12  # eV --> erg
C = constants.c.cgs.value      # Speed of light (cm/s)
H = constants.h.cgs.value      # Planck constant (erg s)
K = constants.k_B.cgs.value    # Boltzmann constant (erg/K)
sigma_SB = constants.sigma_sb.cgs.value  # Stefan-Boltzmann constant in cgs units

Planck function

In [ ]:
def planck_function(wavelength, T):
    """Compute the blackbody radiation intensity using the Planck function
    
    Args:
        wavelength: Wavelength in meters.
        T: Temperature in Kelvin.
    
    Returns:
        Blackbody radiation intensity (energy density per unit wavelength).
    """
    # Avoid division by zero for wavelength or temperature
    wavelength = np.clip(wavelength, 1e-20, None)  # Prevent zero wavelengths
    if T<1e-6: T=1e-6 # Prevent zero or near-zero temperatures

    # Compute the exponential term safely
    exponent = H * C / (wavelength * K * T)
    safe_exponent = np.clip(exponent, None, 700)  # Clamp exponent to avoid overflow

    # Planck function
    B_lambda = (2 * H * C**2) / (wavelength**5) * (1. / (np.exp(safe_exponent) - 1.))
    return B_lambda

The UV component of the ISRF (in Draine unit)

In [ ]:
def radiation_UV(...): # Draine's Field
    ### Your code here

The ISRF, including UV and non-UV components

In [ ]:
def radiation_intensity(...):
   ### Your code here

Radiative heating rate

In [ ]:
def radiative_heating(...):
    #Qabsorption efficiency
    w_um, Qabs = np.loadtxt('Qabs_astrosil_Draine2003.txt', skiprows=5,unpack=True)
    w_cm = w_um * 1e-4  # convert from microns to cm
    
    ### Your code here


Radiative cooling rate

In [ ]:
def radiative_cooling(...):
    ### Your code here

Heating due to the secondary UV photon produced by CRs

In [ ]:
def CR_heating(...):
    ### Your code here

Solve for the equilibrium dust temperature (radiative heating == radiative cooling)

In [ ]:
def Tdust_equilibrium_radiative(a,chi0, Av):
    """Compute the equilibrium dust temperature for a given grain size and radiation field.

    Args:
        a    : Grain radius (cm).
        chi0 : Incident radiation field strength (dimensionless).
        Av   : Visual extinction (mag).
    """
    def func(Td):
        urad_heat = radiative_heating(...)
        urad_cool = radiative_cooling(...)
        return urad_heat - urad_cool

    Td_eq = bisect(func, 2.73, 500.0, xtol=1e-5, maxiter=10000)
    
    return Td_eq

Solve for the equilibrium dust temperature (radiative heating + CR heating == radiative cooling)

In [ ]:
def Tdust_equilibrium_radiative_CR(a, chi0, Av, zeta):
    """Compute the equilibrium dust temperature considering radiative and cosmic ray heating.

    Args:
        a : Grain radius (cm).
        chi0 : Incident radiation field strength (dimensionless).
        Av : Visual extinction (mag).
        zeta : Cosmic ray ionization rate (s^-1).
    """
    def func(Td):
        urad_heat = radiative_heating(...)
        urad_cool = radiative_cooling(...)
        cr_heat = CR_heating(...)
        return urad_heat + cr_heat - urad_cool

    from scipy.optimize import bisect
    Td_eq = bisect(func, 2.73, 500.0, xtol=1e-5, maxiter=10000)
    
    return Td_eq

Cosmic-ray ionization rate as a function of the gas column density

In [ ]:
def zeta_CR(Ngas, model='L'):
    if model == 'L':
        c = np.array([
            -3.331056497233e6,
            1.207744586503e6,
            -1.913914106234e5,
            1.731822350618e4,
            -9.790557206178e2,
            3.543830893824e1,
            -8.034869454520e-1,
            1.048808593086e-2,
            -6.188760100997e-5,
            3.122820990797e-8
        ])
    elif model=='H':
        c = np.array([1.001098610761e7,
                    -4.231294690194e6,
                        7.921914432011e5,
                    -8.623677095423e4,
                    6.015889127529e3,
                    -2.789238383353e2,
                    8.595814402406e0,
                    -1.698029737474e-1,
                    1.951179287567e-3,
                    -9.937499546711e-6
                    ])
    else:
        raise ValueError("Model must be 'L' or 'H'")

    x = np.log10(Ngas)
    log10_zeta= sum(c[k] * x**k for k in range(len(c)))

    zeta = 10**log10_zeta
    return zeta  # s^-1

The dust temperature at equilibrium as a function of Av (without the CR heating)

In [ ]:
if __name__ == "__main__":
    a = 0.1e-4  # cm
    chi0=1.0    # typical ISRF
    Av_range = np.logspace(np.log10(0.01), np.log10(100.0), 100)
        
    # your code here

The dust temperature at equilibrium as a function of Av (with the CR heating)

In [ ]:
if __name__ == "__main__":
    a = 0.1e-4  # cm
    chi0=1.0    # typical ISRF
    Av_range = np.logspace(np.log10(0.01), np.log10(100.0), 100)
    zeta_scale = 1.0 # scale factor for cosmic ray flux
    Ngas_range = 1.6e21 * Av_range  # cm-2
    zeta_range = zeta_CR(Ngas_range, model='L') * zeta_scale  # s^-1

    # your code here
